# Labor forecast workflow conversion

Incremental pandas translation of the Alteryx workflow. This notebook covers the **Current Labor Forecast** and **Prior Labor Forecast** branches through their `Type` filters. Their downstream join and later database outputs will be added as their configurations are captured.

## Current Labor Forecast input

Alteryx Dynamic Input (2) reads the `Data - Forecast & Opps` sheet from the current workbook, treats row 1 as headers, and adds `FileName` as the file name only. The workbook path comes from the existing `.env` key `currentlaborpath`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
current_labor_path = Path(os.environ["currentlaborpath"])
current_labor_raw = pd.read_excel(
    current_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
current_labor_raw["FileName"] = current_labor_path.name
print(len(current_labor_raw), "current labor forecast rows loaded")

## GLC cleanup and Transpose (93)

The Transpose tool keeps the eight descriptive fields and `FileName`, excludes `Labor Forecast Total`, and transposes the selected monthly columns plus dynamic or unknown columns. Missing expected fields raise a clear error for review.

In [ ]:
current_labor_raw["GLC"] = current_labor_raw["GLC"].astype("string").str.strip()

key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_keys = [column for column in key_columns if column not in current_labor_raw.columns]
if missing_keys:
    raise KeyError(f"Missing current forecast key columns: {missing_keys}")

value_columns = [
    column for column in current_labor_raw.columns
    if column not in key_columns and column != "Labor Forecast Total"
]
current_labor = current_labor_raw.melt(
    id_vars=key_columns,
    value_vars=value_columns,
    var_name="Name",
    value_name="Value",
)
print(len(current_labor), "transposed rows from", len(value_columns), "data columns")

## Formula (95)

`DoesHaveMonth` uses `REGEX_Match([Name], ".*\\d+.*")` to identify a digit in the transposed column name. `Segment` uses `Trim([Segment])` and has a 255-character output field.

In [ ]:
current_labor["DoesHaveMonth"] = (
    current_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
current_labor["Segment"] = current_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
current_labor[["Name", "Value", "DoesHaveMonth", "Segment"]].head()

## Parse month and categorize current forecast hours

RegEx (96) parses the first two word tokens from `Name` (for example, `Oct 26` from `Oct 26 Hours`). DateTime (97) parses that result with Alteryx's custom `Mon yy` format to a first-of-month date. Select (103) renames the numeric `Value` field to `NEW Hours`. Text To Columns (318) splits the original name into three words; Formula (100) uses the third word as `Category` when `DoesHaveMonth` is true, otherwise the original `Name`. Select (101) removes the temporary fields; Filter (102) keeps records with non-null `Type`.

Unparseable month labels remain `NaT` for review. The `DoesHaveMonth` indicator is used internally and removed at the final Select.

In [ ]:
# RegEx (96) and DateTime (97)
current_labor["RegExOut1"] = current_labor["Name"].astype("string").str.extract(
    r"(\w+ \w+)", expand=False
)
current_labor["DateTime_Out"] = pd.to_datetime(
    current_labor["RegExOut1"], format="%b %y", errors="coerce"
)

# Select (103) and Text To Columns (318)
current_labor = current_labor.rename(columns={"Value": "NEW Hours"})
current_labor["NEW Hours"] = pd.to_numeric(current_labor["NEW Hours"], errors="coerce")
name_parts = current_labor["Name"].astype("string").str.split(" ", n=2, expand=True)
third_word = name_parts[2] if 2 in name_parts.columns else pd.Series(
    pd.NA, index=current_labor.index, dtype="string"
)

# Formula (100): Alteryx uses -1 for a true RegEx_Match.
current_labor["Category"] = current_labor["Name"].astype("string").where(
    ~current_labor["DoesHaveMonth"], third_word
)

# Select (101) and true output of Filter (102)
current_labor_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
    "NEW Hours", "RegExOut1", "DateTime_Out", "Category",
]
current_labor = current_labor.loc[
    current_labor["Type"].notna(), current_labor_columns
].copy()
current_labor.head()

## Prior Labor Forecast input and Transpose (49)

Filter (176) chooses `Labor Forecast - Prior.xlsx` from the workbook list. Formula (42) selects its `Data - Forecast & Opps` sheet, Dynamic Input (43) reads it, and Formula (408) trims `GLC`. The prior workbook sits beside the current workbook, so its path is derived from the directory of `.env` key `currentlaborpath`. Transpose (49) retains the eight descriptive fields and `FileName`, and unpivots the twelve `Hours` columns from September 2026 through August 2027. `Labor Forecast Total` is excluded. Alteryx is set to warn about missing columns; the notebook warns and skips missing month columns.

In [ ]:
import warnings

prior_labor_path = current_labor_path.with_name("Labor Forecast - Prior.xlsx")
prior_labor_raw = pd.read_excel(
    prior_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
prior_labor_raw["FileName"] = prior_labor_path.name
prior_labor_raw["GLC"] = prior_labor_raw["GLC"].astype("string").str.strip().str.slice(0, 255)

prior_key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_prior_keys = [
    column for column in prior_key_columns if column not in prior_labor_raw.columns
]
if missing_prior_keys:
    raise KeyError(f"Missing prior forecast key columns: {missing_prior_keys}")

prior_month_columns = [
    "Sep 26 Hours", "Oct 26 Hours", "Nov 26 Hours", "Dec 26 Hours",
    "Jan 27 Hours", "Feb 27 Hours", "Mar 27 Hours", "Apr 27 Hours",
    "May 27 Hours", "June 27 Hours", "July 27 Hours", "Aug 27 Hours",
]
missing_prior_months = [
    column for column in prior_month_columns if column not in prior_labor_raw.columns
]
if missing_prior_months:
    warnings.warn(f"Prior forecast month columns missing: {missing_prior_months}")
present_prior_months = [
    column for column in prior_month_columns if column in prior_labor_raw.columns
]
prior_labor = prior_labor_raw.melt(
    id_vars=prior_key_columns,
    value_vars=present_prior_months,
    var_name="Name",
    value_name="Value",
)
print(len(prior_labor), "prior forecast rows from", len(present_prior_months), "month columns")


## Prior month, category, and Type filter

Formula (50) identifies a digit in `Name` and trims `Segment`. RegEx (51) extracts `Mon yy`, DateTime (52) converts it to the first of that month, and Select (58) renames `Value` to `Hours` (numeric). Text To Columns (54) splits `Name` on a space into three fields. Formula (55) uses the third field as `Category` for month headers and the original `Name` otherwise. Select (56) removes temporary fields, and the true output of Filter (57) keeps non-null `Type`.

In [ ]:
# Formula (50), RegEx (51), and DateTime (52)
prior_labor["DoesHaveMonth"] = (
    prior_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
prior_labor["Segment"] = prior_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
prior_labor["RegExOut1"] = prior_labor["Name"].astype("string").str.extract(
    r"(\w+ \w+)", expand=False
)
prior_labor["DateTime_Out"] = pd.to_datetime(
    prior_labor["RegExOut1"], format="%b %y", errors="coerce"
)

# Select (58), Text To Columns (54), and Formula (55)
prior_labor = prior_labor.rename(columns={"Value": "Hours"})
prior_labor["Hours"] = pd.to_numeric(prior_labor["Hours"], errors="coerce")
prior_name_parts = prior_labor["Name"].astype("string").str.split(" ", n=2, expand=True)
prior_third_word = (
    prior_name_parts[2] if 2 in prior_name_parts.columns
    else pd.Series(pd.NA, index=prior_labor.index, dtype="string")
)
prior_labor["Category"] = prior_labor["Name"].astype("string").where(
    ~prior_labor["DoesHaveMonth"], prior_third_word
)

# Select (56) and the true output of Filter (57)
prior_labor_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
    "Hours", "RegExOut1", "DateTime_Out", "Category",
]
prior_labor = prior_labor.loc[
    prior_labor["Type"].notna(), prior_labor_columns
].copy()
prior_labor.head()
